# 00 – Build the shared answer cache (GPU, Kaggle)

Guide Step 0. Every member runs this notebook on **their own Kaggle account** for their part of the cache.

| Owner | Datasets to generate | Extra |
| --- | --- | --- |
| Sai Akhil Singade | `triviaqa_probe`, `triviaqa_pool` | semantic entropy for all models (section 5) |
| Aditya Arora | `bioasq` | – |
| Shivam Vivek Rai | `mmlu_med`, `mmlu_other` | precision-shift runs (section 6) |

**Kaggle settings:** Accelerator = GPU T4 x2 (or P100), Internet = On, Persistence = Files.
Fixed scope: Qwen3-4B and Qwen3-1.7B with 10 samples, Qwen3-8B greedy answers only (no semantic entropy for 8B). Order: 4B first (baseline), then 1.7B, then 8B. Each run is resumable.

In [ ]:
# --- Setup (implementation guide, section 2). Run once per session. ------------------------------
import os, sys, subprocess, random, json
from pathlib import Path

REPO_URL = "https://github.com/adiarora28/dsa1-conformal-abstention.git"   # <- set once for the team
ON_KAGGLE = Path("/kaggle").exists()
REPO_DIR = Path("/kaggle/working/dsa1-conformal-abstention") if ON_KAGGLE else Path.cwd().resolve().parent
if ON_KAGGLE and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if ON_KAGGLE:  # upgrade only what Kaggle's image may lack; never reinstall torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.56",
                    "accelerate", "bitsandbytes", "sentence-transformers", "xgboost"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
except ImportError:
    print("PyTorch not installed (this notebook can run on CPU)")

from dsa1 import config as C
from dsa1.cache_io import link_cache_roots
CACHE_ROOT = link_cache_roots("/kaggle/input", "/kaggle/working/cache_merged") if ON_KAGGLE else Path("cache")
print("CACHE_ROOT:", CACHE_ROOT)
if ON_KAGGLE:  # bring outputs of earlier notebooks (attached as inputs) into this working copy
    import shutil
    Path("results").mkdir(exist_ok=True)
    for p in Path("/kaggle/input").rglob("semantic_entropy"):
        if p.is_dir():
            shutil.copytree(p, "results/semantic_entropy", dirs_exist_ok=True)
    for name in ["hand_check.csv", "failure_cases.csv"]:
        hits = sorted(Path("/kaggle/input").rglob(name))
        if hits and not (Path("results") / name).exists():
            shutil.copy(hits[0], Path("results") / name)
            print("copied", hits[0])
for p in sorted(Path(CACHE_ROOT).glob("*/*")):
    print("  ", p.relative_to(CACHE_ROOT))


In [ ]:
ME = "Sai Akhil"   # <- "Sai Akhil" | "Aditya" | "Shivam"
OWNER_DATASETS = {"Sai Akhil": ["triviaqa_probe", "triviaqa_pool"],
                  "Aditya": ["bioasq"],
                  "Shivam": ["mmlu_med", "mmlu_other"]}
OUT_ROOT = Path("/kaggle/working/cache") if ON_KAGGLE else Path("cache")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

import shutil, subprocess, time
def run(cmd):
    print(">>", cmd)
    subprocess.run(cmd, shell=True, check=True)

# Resume: copy what earlier versions of YOUR cache dataset already contain (attach it as an input first).
for p in Path(CACHE_ROOT).glob("*/*"):
    if p.name in OWNER_DATASETS[ME]:
        dst = OUT_ROOT / p.parent.name / p.name
        if not dst.exists():
            shutil.copytree(p.resolve(), dst)
            print("resumed from input:", dst)

## 1. Five-minute smoke test (do this before any long run)

In [ ]:
run("python scripts/smoke_test.py --model qwen3-1.7b")

## 2. Timing check: 48 questions per model, then an estimate for your whole part

In [ ]:
from dsa1.cache_io import cache_dir, load_records
from dsa1.data import read_questions
MODEL = "qwen3-4b"
SIZES = {"triviaqa_probe": 2000, "triviaqa_pool": 3000, "bioasq": len(read_questions("bioasq")),
         "mmlu_med": 945, "mmlu_other": 2000}
total = 0.0
for model, extra in [("qwen3-4b", ""), ("qwen3-1.7b", ""), ("qwen3-8b", " --no-samples")]:
    for ds in OWNER_DATASETS[ME]:
        run(f"python scripts/build_cache.py --model {model} --dataset {ds} --cache-root {OUT_ROOT} --limit 48{extra}")
        rec = load_records(cache_dir(OUT_ROOT, model, ds))
        per_q = rec[[c for c in ["t_greedy", "t_samples", "t_hidden", "t_forward"] if c in rec]].sum(axis=1).mean()
        hours = per_q * SIZES[ds] / 3600
        total += hours
        print(f"{model} {ds}: {per_q:.2f} s/question -> about {hours:.1f} GPU hours for {SIZES[ds]} questions")
print(f"Total for your part: about {total:.1f} GPU hours. Post this number in the team chat.")

## 3. Full runs

Kaggle stops a session after about 12 hours. The script skips questions already saved, so rerun this cell
in a new session after re-attaching your dataset (section 4) and running the resume cell above.

In [ ]:
# Fixed scope: Qwen3-4B and Qwen3-1.7B with 10 samples (semantic entropy); Qwen3-8B greedy answers only
for model, extra in [("qwen3-4b", ""), ("qwen3-1.7b", ""), ("qwen3-8b", " --no-samples")]:
    for ds in OWNER_DATASETS[ME]:
        run(f"python scripts/build_cache.py --model {model} --dataset {ds} --cache-root {OUT_ROOT}{extra}")

## 4. Publish your part as a Kaggle Dataset

1. Click **Save Version → Save & Run All** (or Quick Save if outputs are already in `/kaggle/working`).
2. Open the saved version → **Output** → **New Dataset** → name it `dsa1-cache-<your part>`
   (`triviaqa`, `bioasq` or `mmlu`), keep it **Private**, then **Settings → Sharing** → add your two teammates.
3. Next time, add a new version of the same dataset instead of creating a new one.

CLI alternative (needs your `kaggle.json` API token as a Kaggle secret):
```
kaggle datasets init -p /kaggle/working/cache        # first time: edit id/title in dataset-metadata.json
kaggle datasets create -p /kaggle/working/cache --dir-mode zip
kaggle datasets version -p /kaggle/working/cache -m "qwen3-8b bioasq" --dir-mode zip
```
Teammates then attach all three `dsa1-cache-*` datasets; `link_cache_roots` merges them automatically.

## 5. Sai Akhil only – semantic entropy (GPU, after all three cache parts are attached)

In [ ]:
if ME == "Sai Akhil":
    for model in ["qwen3-4b", "qwen3-1.7b"]:   # fixed scope: no semantic entropy for Qwen3-8B
        for ds in ["triviaqa_pool", "bioasq"]:
            run(f"python scripts/compute_semantic_entropy.py --model {model} --dataset {ds} --cache-root {CACHE_ROOT}")
    run(f"python scripts/hand_check_sheet.py --cache-root {CACHE_ROOT}")   # 100 answers to label by hand

## 6. Shivam only – precision shift for Task 3 (Qwen3-4B in 8-bit and 4-bit, greedy only)

In [ ]:
if ME == "Shivam":
    for prec in ["8bit", "4bit"]:
        for ds in ["triviaqa_pool", "bioasq"]:
            run(f"python scripts/build_cache.py --model qwen3-4b --dataset {ds} --precision {prec} "
                f"--no-samples --qids-file data/splits/precision_subset.json --cache-root {OUT_ROOT}")

In [ ]:
run("pip freeze | grep -E \"^(torch|transformers|accelerate|bitsandbytes|datasets|sentence-transformers|scikit-learn|xgboost|scipy|numpy|pandas|pyarrow)==\" > /kaggle/working/requirements-lock.txt; cat /kaggle/working/requirements-lock.txt")